# 05 - Exportar a TFLite

Convierte ambos modelos a `.tflite` con cuantizacion int8 dinamica. Verifica tamaño y prueba inferencia.


In [ ]:
!pip install -q tensorflow

In [ ]:
import tensorflow as tf
import numpy as np
from pathlib import Path

OUT = Path("./outputs")

# TFLite NO soporta mixed_float16 — cambiar a float32 para export
original_policy = tf.keras.mixed_precision.global_policy().name
tf.keras.mixed_precision.set_global_policy("float32")

m1 = tf.keras.models.load_model(OUT / "model1_binary.keras")
m2 = tf.keras.models.load_model(OUT / "model2_pathogen.keras")
print("Modelos cargados OK (float32 para export)")


In [ ]:
def export_tflite(model, path, target_mb=None):
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    try:
        tflite = conv.convert()
    except Exception as e:
        print(f"  Error en conversión {path}: {e}")
        print("  Reintentando sin optimizaciones...")
        conv = tf.lite.TFLiteConverter.from_keras_model(model)
        tflite = conv.convert()

    Path(path).write_bytes(tflite)
    size_mb = Path(path).stat().st_size / (1024 * 1024)
    print(f"  {path} - {size_mb:.2f} MB", end="")
    if target_mb:
        print(f"  ({'OK' if size_mb < target_mb else 'GRANDE: revisar'})", end="")
    print()
    return size_mb


export_tflite(m1, OUT / "model1.tflite", target_mb=5)
export_tflite(m2, OUT / "model2.tflite", target_mb=10)

# Restaurar política original (no necesario aquí pero buena practica)
tf.keras.mixed_precision.set_global_policy(original_policy)


In [ ]:
def labels_from_indices(path_in, path_out):
    import json
    with open(path_in) as f:
        idx = json.load(f)
    sorted_labels = [k for k, _ in sorted(idx.items(), key=lambda kv: kv[1])]
    Path(path_out).write_text("\n".join(f"{i} {lbl}" for i, lbl in enumerate(sorted_labels)))
    print(f"  {path_out}: {sorted_labels}")


labels_from_indices(OUT / "class_indices_model1_binary.json", OUT / "labels_m1.txt")
labels_from_indices(OUT / "class_indices_model2_pathogen.json", OUT / "labels_m2.txt")


In [ ]:
# Verificar inferencia M1
inter1 = tf.lite.Interpreter(model_path=str(OUT / "model1.tflite"))
inter1.allocate_tensors()
inp1 = inter1.get_input_details()[0]
out1 = inter1.get_output_details()[0]
print("M1 Input:", inp1["shape"], inp1["dtype"])
print("M1 Output:", out1["shape"], out1["dtype"])
dummy = np.random.rand(1, 224, 224, 3).astype(np.float32)
inter1.set_tensor(inp1["index"], dummy)
inter1.invoke()
print("M1 salida prueba:", inter1.get_tensor(out1["index"]))

# Verificar inferencia M2
inter2 = tf.lite.Interpreter(model_path=str(OUT / "model2.tflite"))
inter2.allocate_tensors()
inp2 = inter2.get_input_details()[0]
out2 = inter2.get_output_details()[0]
print("M2 Input:", inp2["shape"], inp2["dtype"])
print("M2 Output:", out2["shape"], out2["dtype"])
inter2.set_tensor(inp2["index"], dummy)
inter2.invoke()
print("M2 salida prueba:", inter2.get_tensor(out2["index"]))


## Copiar a Code/

```
Code/assets/models/hs/model.tflite          <- outputs/model1.tflite
Code/assets/models/hs/labels.txt            <- outputs/labels_m1.txt
Code/assets/models/pd/model_unquant.tflite  <- outputs/model2.tflite
Code/assets/models/pd/labels.txt            <- outputs/labels_m2.txt
```
